# QuizForge – Question Generation Demo
**Stages:** text extraction → section-aware passages → answer candidates → Seq2Seq LLM (Flan-T5) writes the question → grounding checks → keywords

## 1. Install requirements

In [1]:
!pip install -q -r ../requirements.txt


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports and project path

In [2]:
import sys, os
sys.path.append(os.path.abspath(".."))   # so `import qgen` works from notebooks/

from qgen.extractor import extract_text, build_chunks
from qgen.candidates import plan_slots
from qgen.generator import QuestionGenerator
from qgen.pipeline import generate_quiz

E:\ai\envs\main\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Load a textbook (PDF or DOCX)
Set `BOOK_PATH` to your own file (keep it under 5 MB, same as the app).

In [3]:
BOOK_PATH = "D:\MIT-AoE\Academics\TY\DAA\Study Material\DAA_Unit_2\Genetic_Optimization_Topics.pdf"   ##

with open(BOOK_PATH, "rb") as f:
    raw_text = extract_text(BOOK_PATH, f.read())   # keep the line breaks: they are used to find headings
print(len(raw_text.split()), "words extracted")

<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
C:\Users\Om Channawar\AppData\Local\Temp\ipykernel_27876\2804201549.py:1: SyntaxWarning: invalid escape sequence '\M'
  BOOK_PATH = "D:\MIT-AoE\Academics\TY\DAA\Study Material\DAA_Unit_2\Genetic_Optimization_Topics.pdf"   ##


5372 words extracted


## 4. Section-aware passages
Every passage stays inside one section and has a title, so the model never mixes unrelated topics.

In [4]:
chunks = build_chunks(raw_text)
print(len(chunks), "passages")
for c in chunks[:5]:
    print(f"[{c.title}] ({len(c.text.split())} words)  {c.text[:100]}...")

58 passages
[Genetic Algorithm (GA)] (134 words)  Genetic Algorithm (GA) is a search-based optimization technique based on the principles of Genetics ...
[Genetic Algorithm (GA)] (86 words)  Selection: The process of choosing chromosomes from the current population based on their fitness, t...
[Gene] (71 words)  Definition: An individual is a single solution in the population. It represents a candidate answer t...
[Fitness] (60 words)  Definition: Fitness is a measure of how good an individual (solution) is. It is evaluated using a fi...
[Population] (114 words)  Definition: The population is a group of individuals (solutions) that evolve over generations. The p...


## 5. Load the Seq2Seq Transformer LLM

In [5]:
generator = QuestionGenerator("google/flan-t5-base")
print("Running on:", generator.device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 282/282 [00:01<00:00, 173.81it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Running on: cuda


## 6. Generate questions
`DIFFICULTY` = `low` / `mid` / `high`, `NUM_QUESTIONS` = multiple of 5 (max 30). **Pass `raw_text`, not cleaned text.**

In [6]:
DIFFICULTY = "mid"
NUM_QUESTIONS = 5

items = generate_quiz(raw_text, generator, DIFFICULTY, NUM_QUESTIONS,
                      progress_callback=lambda p: print(f"\rprogress: {p:.0%}", end=""))
print()
for i, it in enumerate(items, 1):
    print(f"Q{i}. {it.question}")

progress: 100%
Q1. What is Genetic Algorithm?
Q2. What is the purpose of crossover?
Q3. What are the characteristics of swarm intelligence?
Q4. What is the process called when particles move toward better solutions?
Q5. What is the performance of BCO?


## 7. Answers, source sentences and keywords

In [7]:
for i, it in enumerate(items, 1):
    print(f"Q{i}. {it.question}")
    print("   Keywords:", ", ".join(it.keywords))
    print("   Answer  :", it.answer)
    print("   Source  :", it.evidence, "\n")

Q1. What is Genetic Algorithm?
   Keywords: technique, principles, selection, search-based, natural, based
   Answer  : Genetic Algorithm (GA) is a search-based optimization technique based on the principles of Genetics and Natural Selection.
   Source  : Genetic Algorithm (GA) is a search-based optimization technique based on the principles of Genetics and Natural Selection. 

Q2. What is the purpose of crossover?
   Keywords: job, precedence, order-based, exchanged, constraints, commonly
   Answer  : For JSSP, order-based crossover is commonly used, where parts of the job sequences are exchanged between parents to create new schedules while maintaining job precedence constraints.
   Source  : For JSSP, order-based crossover is commonly used, where parts of the job sequences are exchanged between parents to create new schedules while maintaining job precedence constraints. 

Q3. What are the characteristics of swarm intelligence?
   Keywords: predators, navigating, cooperate, finding,

## 8. (Optional) What the planner picks at each difficulty

In [8]:
for level in ["low", "mid", "high"]:
    slots = plan_slots(chunks, 5, level)
    first = slots[0][0]
    print(level.upper(), "| answer:", first.answer[:80])
    print("      question:", generator.generate_question(first, level), "\n")

LOW | answer: Genetic Algorithm (GA)
      question: What is a search-based optimization technique based on? 

MID | answer: Genetic Algorithm (GA) is a search-based optimization technique based on the pri
      question: What is Genetic Algorithm? 

HIGH | answer: Genetic Algorithm (GA) is a search-based optimization technique based on the pri
      question: What is Genetic Algorithm? 

